# ETL COVID historical series

## 1. Load raw dataset

In [0]:
CATALOG = "workspace"
SCHEMA = "default"
TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"

RAW_TABLE = f"{TABLE_PREFIX}.raw_covid"
OUTPUT_TABLE = f"{TABLE_PREFIX}.fato_covid"

df_raw = spark.table(RAW_TABLE)

display(df_raw)
print(df_raw.columns)

## 2. Define helper functions and month order

In [0]:
from functools import reduce
from pyspark.sql.functions import (
    col, lit, trim, upper, regexp_replace, when, create_map, element_at
)

VALID_MONTHS = [
    "JANEIRO", "FEVEREIRO", "MARÇO", "ABRIL", "MAIO", "JUNHO",
    "JULHO", "AGOSTO", "SETEMBRO", "OUTUBRO", "NOVEMBRO", "DEZEMBRO"
]

MONTH_ORDER = {
    "JANEIRO": 1,
    "FEVEREIRO": 2,
    "MARÇO": 3,
    "ABRIL": 4,
    "MAIO": 5,
    "JUNHO": 6,
    "JULHO": 7,
    "AGOSTO": 8,
    "SETEMBRO": 9,
    "OUTUBRO": 10,
    "NOVEMBRO": 11,
    "DEZEMBRO": 12,
}

month_order_map = create_map(*[x for item in MONTH_ORDER.items() for x in (lit(item[0]), lit(item[1]))])

def clean_int(column_name):
    cleaned = regexp_replace(trim(col(column_name).cast("string")), r"[^0-9-]", "")
    return when((cleaned == "") | cleaned.isNull(), None).otherwise(cleaned.cast("int"))

## 3. Select valid month rows

In [0]:
df_covid_months = (
    df_raw
    .withColumn("mes", regexp_replace(trim(col("_c0").cast("string")), "^\ufeff", ""))
    .withColumn("mes", upper(col("mes")))
    .filter(col("mes").isin(VALID_MONTHS))
)

display(df_covid_months.select("mes", "_c1", "_c2", "_c13", "_c14"))

## 4. Transform from wide to long format

In [0]:
YEAR_COLUMNS = {
    2020: ("_c1", "_c2"),
    2021: ("_c3", "_c4"),
    2022: ("_c5", "_c6"),
    2023: ("_c7", "_c8"),
    2024: ("_c9", "_c10"),
    2025: ("_c11", "_c12"),
    2026: ("_c13", "_c14"),
}

dfs = []

for year, (cases_col, deaths_col) in YEAR_COLUMNS.items():
    df_year = (
        df_covid_months
        .select(
            col("mes"),
            lit(year).alias("ano"),
            clean_int(cases_col).alias("casos"),
            clean_int(deaths_col).alias("obitos"),
            lit("covid").alias("doenca")
        )
    )
    dfs.append(df_year)

df_covid_long = reduce(lambda a, b: a.unionByName(b), dfs)

display(df_covid_long.orderBy("ano", "mes"))

## 5. Clean, filter unpublished months, and add chronological order

In [0]:
df_covid_final = (
    df_covid_long
    .filter(col("casos").isNotNull())  # removes months not published yet, for example May-Dec/2026
    .withColumn("obitos", when(col("obitos").isNull(), lit(0)).otherwise(col("obitos")))
    .withColumn("ordem_mes", element_at(month_order_map, col("mes")))
    .select("ano", "mes", "ordem_mes", "casos", "obitos", "doenca")
    .orderBy("ano", "ordem_mes")
)

display(df_covid_final)
df_covid_final.printSchema()

## 6. Save transformed table

In [0]:
df_covid_final.write     .format("delta")     .mode("overwrite")     .option("overwriteSchema", "true")     .saveAsTable(OUTPUT_TABLE)

## 7. Validate

In [0]:
df_saved = spark.table(OUTPUT_TABLE)

print("Rows:", df_saved.count())

display(
    df_saved
    .groupBy("ano")
    .sum("casos", "obitos")
    .orderBy("ano")
)

display(df_saved.orderBy("ano", "ordem_mes"))